# 52. Agents, workflows, and bounded tool use

**Level:** Advanced

**Prerequisites:** 40-43: prompts, tools, retrieval, and evaluations.

[Course index](00_START_HERE.ipynb) | [Previous](51_Time_Series_and_Forecasting.ipynb) | [Next](53_Capstone_Network_Telemetry_Classifier.ipynb)

**Topics:** Agents, Workflows, Tool calling, Planning, Budgets, Approval, Prompt injection.

**Matching script:** `examples\52_Agents_Workflows_and_Bounded_Tools.py`

**How to study:** read the notes, predict an example's output, run it, trace each change in data/shape/state, then complete the practice task. Every worked code cell contains its own imports and data and can run independently of other notebooks.

**Execution contract:** Python 3.11+, course dependencies, CPU only, synthetic/local data. No hosted API requests, model-weight downloads, dataset downloads, listening web servers, credentials, or real device changes. The temporary-file lesson cleans up its own files. Expected errors are caught narrowly and asserted; unexpected errors must remain visible.

## Concept notes and mental model

A workflow uses predefined control flow to coordinate operations. An agent lets a model choose
actions dynamically based on observations. A loop with a fixed scripted planner is a useful
workflow demonstration but is not evidence of autonomous language-model planning.

Tools expand what a system can do and therefore expand risk. Define allowed operations, scoped
inputs, authorization, timeouts, step/cost budgets, and explicit stop conditions. Treat external
documents and tool output as untrusted data. Prompt injection can try to turn those inputs into
instructions; an instruction saying 'ignore injection' is not sufficient isolation.

For consequential actions, require application-level approval bound to the exact proposed action.
Keep read-only and write tools separate, log outcomes without leaking secrets, and inspect
failures rather than reporting success after an exhausted budget.

Use the simplest reliable design. Measure task completion and side effects on representative
cases before adding autonomy or multiple agents. The example below uses a deterministic mock
planner and a local read-only inventory tool, with a hard step limit.

## Mathematics or data contract

A bounded loop is propose -> validate/authorize -> execute -> observe -> stop, with a maximum number of steps.

## Important limitation

The planner is scripted, not an LLM; no external access or autonomous production action occurs.

Distinguish what is implemented and asserted below from what would need additional data, training, independent evaluation, authorization, or operational controls.

## Worked example 1: A bounded mock planner with a read-only tool

**Purpose:** Prove budget and allowlist behavior.

### Read the code in this order
1. Keep the planner's proposals separate from execution.
2. Allow only a read-only local tool.
3. Pass observed results back to the planner.
4. Raise if the budget is exhausted instead of fabricating completion.

### Syntax and behavior to notice
- `assert` checks the stated behavior; a failed check raises AssertionError.
- `raise` reports invalid input or a failed requirement instead of fabricating success.

### Expected output
This is the output captured from executing the worked example in the recorded environment. Package versions and floating-point rounding can change numeric details; the assertions express the required shapes, invariants, and thresholds.

```text
edge-01 is up tool calls: 1
```

**Before running:** identify the input variables, which values are learned versus chosen, and what each assertion proves. A successful shape test does not by itself prove predictive quality. Training-only loss checks are labeled as such.

In [1]:
def inventory_status(hostname):
    known = {"edge-01": "up"}
    return {"hostname": hostname, "state": known[hostname]}

def mock_plan(observations):
    if not observations:
        return {"tool": "inventory_status", "arguments": {"hostname": "edge-01"}}
    return {"final": f"edge-01 is {observations[-1]['state']}"}

tools = {"inventory_status": inventory_status}
observations, final = [], None
for step in range(3):
    proposal = mock_plan(observations)
    if "final" in proposal:
        final = proposal["final"]
        break
    name = proposal["tool"]
    if name not in tools:
        raise PermissionError("tool not allowed")
    arguments = proposal["arguments"]
    if set(arguments) != {"hostname"} or arguments["hostname"] != "edge-01":
        raise PermissionError("target not authorized")
    observations.append(tools[name](**arguments))
else:
    raise RuntimeError("step budget exhausted")
assert final == "edge-01 is up" and len(observations) == 1
print(final, "tool calls:", len(observations))

edge-01 is up tool calls: 1


### Interpret example 1

The result should demonstrate this purpose: **Prove budget and allowlist behavior.**

Follow the assertions back to the exact data or state being checked. Where a model was fitted, distinguish training fit from held-out evaluation. Where input is rejected, inspect whether rejection happens before any intended mutation or tool execution. Do not delete a failed assertion to make an incorrect result look valid.

## Practice: extend the concept yourself

Make the mock planner request an unknown tool and assert rejection before any tool runs.

1. Write your expected result before editing code.
2. Add one normal-case test and one boundary/failure-case test.
3. Keep training/validation/test responsibilities separate where relevant.
4. Report an honest failure or unknown result rather than a success-shaped fallback.

The task is intentionally not filled in for you; the worked examples above are complete and runnable. Add your own code cell for the extension.

## Understanding checkpoint
- Explain `Agents` in your own words and give one example of when it is useful.
- Trace one worked example from input through transformation to returned or printed output.
- State this lesson's mathematical/data contract: A bounded loop is propose -> validate/authorize -> execute -> observe -> stop, with a maximum number of steps.
- Name the boundary of the demonstration: The planner is scripted, not an LLM; no external access or autonomous production action occurs.
- Describe one test that could catch a plausible but wrong implementation.

## Primary references and further reading

- `https://www.anthropic.com/engineering/building-effective-agents`
- `https://www.anthropic.com/engineering/demystifying-evals-for-ai-agents`

Research snapshot: **2026-10-17**. Foundational references need not be recent. Mutable library docs describe their current version; compare with the tested dependency versions in the course manifest. References are reading material, not runtime downloads.